In [1]:
from pathlib import Path
import pandas as pd
import mlflow
import plotly.express as px
from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss
import numpy as np

## Load reference submission

In [2]:
CLASSES = [
    "antelope_duiker", "bird", "blank", "civet_genet",
    "hog", "leopard", "monkey_prosimian", "rodent",
]
prob_cols = [f"{c}_prob" for c in CLASSES]
cls2idx = {c: i for i, c in enumerate(CLASSES)}

In [3]:
example = '../data/competition/submission_format.csv'
ex = pd.read_csv(example)

In [4]:
print(ex.columns)
ex.head()

Index(['id', 'antelope_duiker', 'bird', 'blank', 'civet_genet', 'hog',
       'leopard', 'monkey_prosimian', 'rodent'],
      dtype='object')


,id,antelope_duiker,bird,blank,civet_genet,hog,leopard,monkey_prosimian,rodent
0,ZJ016488,0.048233,0.189185,0.044914,0.199588,0.106118,0.132915,0.166410,0.112637
1,ZJ016489,0.097078,0.061400,0.026409,0.241530,0.144344,0.051780,0.287811,0.089648
2,ZJ016490,0.124658,0.089101,0.189225,0.174494,0.180540,0.079995,0.085672,0.076314
3,ZJ016491,0.109966,0.048397,0.055598,0.323600,0.322356,0.063252,0.008160,0.068671
4,ZJ016492,0.165742,0.184610,0.005431,0.136806,0.000389,0.122078,0.151521,0.233423


In [5]:
output_dir = Path("submissions_3.12.26")
output_dir.mkdir(parents=True, exist_ok=True)

## Load mlflow tracking

In [6]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

In [7]:
experiment = mlflow.get_experiment_by_name("conservision")
if experiment:
    experiment_id = experiment.experiment_id
    # Load all runs from the experiment into a pandas DataFrame
    runs_df = mlflow.search_runs(experiment_ids=[experiment_id])
else:
    print("Experiment not found.")

2026/03/12 11:41:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/12 11:41:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/12 11:41:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/12 11:41:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/12 11:41:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/12 11:41:37 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/12 11:41:38 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/12 11:41:38 INFO alembic.runtime.migration: Will assume non-transactional DDL.


In [8]:
experiment = mlflow.get_experiment_by_name("conservision_ovr")
if experiment:
    experiment_id = experiment.experiment_id
    # Load all runs from the experiment into a pandas DataFrame
    ovr_df = mlflow.search_runs(experiment_ids=[experiment_id])
else:
    print("Experiment not found.")

In [9]:
runs_df

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.lr,metrics.val_log_loss,metrics.f1_bird,metrics.f1_civet_genet,...,params.train_samples,params.patience,params.num_workers,params.class_weights.monkey_prosimian,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.git.commit,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.runColor
0,61b2042dd74447c2abbaf573951d0f4d,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/61b2...,2026-03-12 08:05:18.802000+00:00,2026-03-12 09:19:23.982000+00:00,0.000003,1.3307,0.6465,0.8185,...,14788,10,4,0.7493,LOCAL,vit_so150_fold4,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train.py,dxd,None
1,79bea0cfa1134725a94eacc064f15bdd,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/79be...,2026-03-12 06:38:46.498000+00:00,2026-03-12 08:04:39.794000+00:00,0.000003,0.8917,0.7644,0.8586,...,14789,10,4,0.7813,LOCAL,vit_so150_fold3,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train.py,dxd,None
2,3e64b69e1ad44e979996fee939a04ac3,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/3e64...,2026-03-12 05:18:02.588000+00:00,2026-03-12 06:38:07.409000+00:00,0.000003,1.3915,0.4150,0.7561,...,14788,10,4,0.7692,LOCAL,vit_so150_fold2,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train.py,dxd,None
3,9a24a29413d34d5c9dd4bb5fe71fbe17,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/9a24...,2026-03-12 04:03:05.205000+00:00,2026-03-12 05:17:23.268000+00:00,0.000003,1.6226,0.5731,0.6395,...,14787,10,4,0.8425,LOCAL,vit_so150_fold1,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train.py,dxd,None
4,7fe539e966624303842728375212236b,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/7fe5...,2026-03-12 02:36:20.507000+00:00,2026-03-12 04:02:25.586000+00:00,0.000003,0.9711,0.8326,0.8861,...,14788,10,4,0.7365,LOCAL,vit_so150_fold0,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train.py,dxd,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,9b7cfe747ae1486d850d97561775faa4,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/9b7c...,2026-02-25 01:28:51.217000+00:00,2026-02-25 02:15:47.297000+00:00,0.000006,0.9777,0.6513,0.8727,...,13959,10,4,0.764,LOCAL,convnext_base_0224_1828,2b812ed2628d670cf414fa13b72b3d63f3f6b9cb,scripts/03_train.py,dxd,None
133,952095a9c2ef4914906f50dfc6663a68,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/9520...,2026-02-24 22:03:16.799000+00:00,2026-02-24 22:43:40.570000+00:00,0.000000,1.3604,0.4786,0.8176,...,13959,5,4,0.764,LOCAL,effnetv2s_0224_1503,2b812ed2628d670cf414fa13b72b3d63f3f6b9cb,scripts/03_train.py,dxd,None
134,c38eb9c9b663441988a8f287895682d3,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/c38e...,2026-02-24 20:11:09.341000+00:00,2026-02-24 20:33:03.505000+00:00,0.000006,0.9423,0.6306,0.8798,...,13959,10,4,0.764,LOCAL,dinov2_vits14_v3_0224_1311,2b812ed2628d670cf414fa13b72b3d63f3f6b9cb,scripts/03_train.py,dxd,None
135,33ff66a9bf104759b634c6baf1c3c86b,1,FINISHED,/home/dxd/Documents/conservision/mlruns/1/33ff...,2026-02-24 19:45:25.705000+00:00,2026-02-24 20:04:32.274000+00:00,0.000008,1.0033,0.4955,0.8806,...,13959,7,4,0.764,LOCAL,dinov2_vits14_v2_0224_1245,2b812ed2628d670cf414fa13b72b3d63f3f6b9cb,scripts/03_train.py,dxd,None


In [10]:
ovr_df

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.val_acc,metrics.lr,metrics.val_log_loss,metrics.training_time_s,...,params.val_fold,params.class_weighting,params.class_weights.rodent,params.timm_create_kwargs.img_size,params.class_weights.leopard,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.git.commit,tags.mlflow.source.name,tags.mlflow.user
0,bf385d7c8cc9476a85c3825191d1c231,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/bf38...,2026-03-10 20:16:08.961000+00:00,2026-03-10 20:54:21.674000+00:00,0.743100,3.456709e-06,0.6078,2275.285105,...,4,inverse_frequency,None,None,None,LOCAL,ovr_eva02_blank_full_fold4,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
1,a234cf2180354ec78f7ff4e0b50998cc,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/a234...,2026-03-10 19:24:40.352000+00:00,2026-03-10 20:15:37.406000+00:00,0.747423,2.173685e-06,0.5203,3039.429982,...,3,inverse_frequency,None,None,None,LOCAL,ovr_eva02_blank_full_fold3,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
2,8470be1e53e74d8dbb9d274095c95a23,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/8470...,2026-03-10 18:39:38.519000+00:00,2026-03-10 19:24:08.900000+00:00,0.654231,2.826315e-06,0.3816,2652.401291,...,2,inverse_frequency,None,None,None,LOCAL,ovr_eva02_blank_full_fold2,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
3,dc1629b17b044e599f56fe8b8b46f6d9,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/dc16...,2026-03-10 17:51:30.478000+00:00,2026-03-10 18:39:07.314000+00:00,0.757126,2.500000e-06,0.6507,2839.528081,...,1,inverse_frequency,None,None,None,LOCAL,ovr_eva02_blank_full_fold1,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
4,a7ba6ffec72d4a93b36691393a7c6ba3,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/a7ba...,2026-03-10 16:53:43.247000+00:00,2026-03-10 17:50:59.097000+00:00,0.845664,1.543291e-06,0.4364,3418.226329,...,0,inverse_frequency,None,None,None,LOCAL,ovr_eva02_blank_full_fold0,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
5,dc1c0371a3714a538b306c0b11a07a99,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/dc1c...,2026-03-10 07:27:37.860000+00:00,2026-03-10 08:18:47.758000+00:00,0.742190,4.347369e-06,0.5562,3053.670592,...,4,inverse_frequency,None,None,None,LOCAL,ovr_blank_convnext_full_fold4,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
6,dc640f2e3b434e248ce37dbccb5f1677,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/dc64...,2026-03-10 06:42:21.907000+00:00,2026-03-10 07:27:08.465000+00:00,0.714676,5.652631e-06,0.6251,2670.217168,...,3,inverse_frequency,None,None,None,LOCAL,ovr_blank_convnext_full_fold3,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
7,e48a4678103841f9a1471a9ad41b1c6c,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/e48a...,2026-03-10 05:47:22.035000+00:00,2026-03-10 06:41:52.839000+00:00,0.770094,3.705905e-06,0.4151,3254.461742,...,2,inverse_frequency,None,None,None,LOCAL,ovr_blank_convnext_full_fold2,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
8,be66946e9a044770a81b45102bcd44e2,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/be66...,2026-03-10 05:05:16.493000+00:00,2026-03-10 05:46:52.999000+00:00,0.722559,6.294095e-06,0.4231,2479.916681,...,1,inverse_frequency,None,None,None,LOCAL,ovr_blank_convnext_full_fold1,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd
9,5554ee403f47425e865e2a9aad3ed275,2,FINISHED,/home/dxd/Documents/conservision/mlruns/2/5554...,2026-03-10 04:10:09.775000+00:00,2026-03-10 05:04:46.396000+00:00,0.821710,3.705905e-06,0.4024,3260.095658,...,0,inverse_frequency,None,None,None,LOCAL,ovr_blank_convnext_full_fold0,afdb8d5231f86d731e66bedd342240e02510ff58,scripts/02_train_ovr.py,dxd


## Check model level pres available

In [11]:
base_model_dir = Path('../models/')
model_folder_avg_test_path = 'averaged_fold_test_preds.csv'

In [12]:
def create_model_agg_preds(model_list, folds=5):
    cols_to_keep = ['antelope_duiker_prob', 'bird_prob', 'blank_prob', 'civet_genet_prob',
       'hog_prob', 'leopard_prob', 'monkey_prosimian_prob', 'rodent_prob']
    for mdl in model_list:
        agg_probas = pd.DataFrame(ex['id'].copy()).reset_index()
        for col in cols_to_keep:
            agg_probas[col] = 0
        test_counts = 0
        
        for i in range(folds):
            fold_dir = mdl / f"fold_{i}"
            fold_test_preds = fold_dir / "predictions_test.csv"
            if fold_test_preds.is_file():
                test_counts += 1

            fold_preds = pd.read_csv(fold_test_preds)
            print("checking id order:\t")
            are_equal_order = agg_probas['id'].equals(fold_preds['id'])
            print(are_equal_order)
            agg_probas[cols_to_keep] = agg_probas[cols_to_keep] + fold_preds[cols_to_keep]
        print(mdl)
        agg_probas[cols_to_keep] = agg_probas[cols_to_keep] / folds
        
        agg_test_out = mdl / "averaged_fold_test_preds.csv"
        agg_probas.to_csv(agg_test_out, index=False)

In [13]:
def check_model_dir(base_dir, target_file):
    skip_list = ['blank', 'archive', '.json']
    for sub_dir in base_dir.iterdir():
        model_avg_test_file = sub_dir / target_file
        if not model_avg_test_file.is_file():
            if any(sl in str(sub_dir) for sl in skip_list):
                pass
            else:
                print(str(sub_dir))
                create_model_agg_preds([sub_dir], folds=5)  

In [14]:
check_model_dir(base_model_dir, model_folder_avg_test_path)

../models/dino_v3_.05_folds
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
../models/dino_v3_.05_folds
../models/eva02_.1_b32_folds
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
../models/eva02_.1_b32_folds
../models/convnext_b32_folds
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
../models/convnext_b32_folds
../models/vit_so150_folds
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
../models/vit_so150_folds
../models/dinov2_vitb14_reg4_folds
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
checking id order:	
True
../models/dinov2_vitb14_reg4_folds
../models/eva02_.05_folds
checking id order:	
True
checking id order:	
True
check

In [15]:
models_list = [
    "convnext_.1_folds",
    "convnext_b32_folds",
    "dinov2_vitb14_reg4_folds",
    "dino_v3_.05_folds",
    "eva02_.1_b32_folds",
    "eva02_.05_folds",
    "swinv2_.1_folds",
    "swinv2_full_folds",
    "vit_so150_folds"
]

In [16]:
from scipy.stats import rankdata

In [17]:
model_paths_ensemble = [Path(f'../models/{mdl}/averaged_predictions_test.csv') for mdl in models_list]
model_paths_ensemble

[PosixPath('../models/convnext_.1_folds/averaged_predictions_test.csv'),
 PosixPath('../models/convnext_b32_folds/averaged_predictions_test.csv'),
 PosixPath('../models/dinov2_vitb14_reg4_folds/averaged_predictions_test.csv'),
 PosixPath('../models/dino_v3_.05_folds/averaged_predictions_test.csv'),
 PosixPath('../models/eva02_.1_b32_folds/averaged_predictions_test.csv'),
 PosixPath('../models/eva02_.05_folds/averaged_predictions_test.csv'),
 PosixPath('../models/swinv2_.1_folds/averaged_predictions_test.csv'),
 PosixPath('../models/swinv2_full_folds/averaged_predictions_test.csv'),
 PosixPath('../models/vit_so150_folds/averaged_predictions_test.csv')]

In [18]:
preds = [pd.read_csv(f) for f in model_paths_ensemble]

In [20]:
def rank_avg(list_dfs):
    agg_probas = pd.DataFrame(ex['id'].copy()).reset_index()
    for col in CLASSES:
        agg_probas[col] = 0
    
    for pred_df in list_dfs:
        pred_df.columns = pred_df.columns.str.removesuffix("_prob")

        # test_preds = f"../models/{p}/averaged_fold_test_preds.csv"
        # pred_df = pd.read_csv(test_preds)
        # print(pred_df.shape)

        agg_probas[CLASSES] = agg_probas[CLASSES] + pred_df[CLASSES].rank(axis=1)
    print(agg_probas[CLASSES].mean())
    agg_probas[CLASSES] = agg_probas[CLASSES] / len(list_dfs)
    print(agg_probas[CLASSES].mean())
    return agg_probas

In [21]:
ranked_df = rank_avg(preds)

antelope_duiker     52.260753
bird                40.449149
blank               48.904122
civet_genet         28.037970
hog                 30.144937
leopard             39.126120
monkey_prosimian    44.179884
rodent              40.897065
dtype: float64
antelope_duiker     5.806750
bird                4.494350
blank               5.433791
civet_genet         3.115330
hog                 3.349437
leopard             4.347347
monkey_prosimian    4.908876
rodent              4.544118
dtype: float64


In [22]:
ranked_df.head()

,index,id,antelope_duiker,bird,blank,civet_genet,hog,leopard,monkey_prosimian,rodent
0,0,ZJ016488,4.111111,4.222222,5.111111,7.888889,2.222222,6.666667,1.555556,4.222222
1,1,ZJ016489,7.000000,4.000000,5.111111,3.444444,4.888889,1.111111,3.222222,7.222222
2,2,ZJ016490,7.555556,4.111111,4.555556,5.777778,6.000000,2.666667,2.444444,2.888889
3,3,ZJ016491,4.222222,3.000000,5.555556,4.222222,5.888889,8.000000,2.111111,3.000000
4,4,ZJ016492,5.444444,6.111111,3.777778,4.666667,1.666667,1.555556,4.777778,8.000000


In [23]:
from scipy.special import softmax

# Apply softmax row-wise to the target columns
# Use .loc to select the columns and ensure correct assignment
ranked_df.loc[:, CLASSES] = softmax(ranked_df[CLASSES].values, axis=1)

print("\nDataFrame after applying softmax to target columns:")
print(ranked_df)

# Verify that the sum of the target columns for each row is 1
ranked_df['sum_check'] = ranked_df[CLASSES].sum(axis=1)
print("\nSum check (target columns sum to 1):")
print(ranked_df['sum_check'])



DataFrame after applying softmax to target columns:
      index        id  antelope_duiker      bird     blank  civet_genet  \
0         0  ZJ016488         0.015929  0.017801  0.043299     0.696387   
1         1  ZJ016489         0.380925  0.018965  0.057611     0.010881   
2         2  ZJ016490         0.673511  0.021500  0.033532     0.113832   
3         3  ZJ016491         0.018012  0.005306  0.068333     0.018012   
4         4  ZJ016492         0.058716  0.114362  0.011090     0.026975   
...     ...       ...              ...       ...       ...          ...   
4459   4459  ZJ020947         0.208551  0.054974  0.633523     0.000806   
4460   4460  ZJ020948         0.546695  0.011190  0.017452     0.042450   
4461   4461  ZJ020949         0.134757  0.001134  0.031786     0.188068   
4462   4462  ZJ020950         0.609123  0.008933  0.024284     0.000775   
4463   4463  ZJ020951         0.052822  0.015560  0.017388     0.037848   

           hog   leopard  monkey_prosimian    

In [24]:
ranked_df.drop(columns=['sum_check'], inplace=True)

In [25]:
ranked_df.drop(columns=['index'], inplace=True)

In [26]:
ranked_df

,id,antelope_duiker,bird,blank,civet_genet,hog,leopard,monkey_prosimian,rodent
0,ZJ016488,0.015929,0.017801,0.043299,0.696387,0.002409,0.205138,0.001237,0.017801
1,ZJ016489,0.380925,0.018965,0.057611,0.010881,0.046131,0.001055,0.008713,0.475718
2,ZJ016490,0.673511,0.021500,0.033532,0.113832,0.142159,0.005071,0.004061,0.006333
3,ZJ016491,0.018012,0.005306,0.068333,0.018012,0.095367,0.787482,0.002181,0.005306
4,ZJ016492,0.058716,0.114362,0.011090,0.026975,0.001343,0.001202,0.030146,0.756166
...,...,...,...,...,...,...,...,...,...
4459,ZJ020947,0.208551,0.054974,0.633523,0.000806,0.003059,0.018097,0.076722,0.004269
4460,ZJ020948,0.546695,0.011190,0.017452,0.042450,0.002950,0.001514,0.027218,0.350530
4461,ZJ020949,0.134757,0.001134,0.031786,0.188068,0.511223,0.009363,0.003082,0.120586
4462,ZJ020950,0.609123,0.008933,0.024284,0.000775,0.082436,0.004587,0.250418,0.019445


In [28]:
format_string = '%.6f'
ranked_df.to_csv('../submissions/submission_3.12.26_1.csv', index=False, float_format=format_string)